# 第 31 章：卷积神经网络与图像局部特征

这个 notebook 用一个极小的星系 cutout 教学数据，把进入 CNN 之前最关键的直觉讲清楚：

- 为什么 raw pixels 直接摊平会对位置变化敏感
- 为什么卷积核更擅长抓局部模式
- 为什么 ReLU 和池化能让表示更稳健
- 怎样把这个教学版卷积 workflow 接到后续真正的 CNN 与迁移学习
- 为什么冻结局部特征提取器再接一个小型 target head，是迁移学习里很常见的第一步

教学说明：这里继续保持仓库当前风格，默认不依赖 NumPy、PyTorch 或图像库；环境里如果装了 `torch`，最后一格会给出一个可选的极小 CNN 对照。


In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path

DATA_PATH = Path("../../data/small/cnn_cutout_demo.csv").resolve()
IMAGE_SIZE = 6

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "morphology_label": raw["morphology_label"],
            "split": raw["split"],
        }
        for row_index in range(IMAGE_SIZE):
            for col_index in range(IMAGE_SIZE):
                parsed[f"p{row_index}{col_index}"] = float(raw[f"p{row_index}{col_index}"])
        rows.append(parsed)

train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]


def image_from_row(row):
    return [[row[f"p{row_index}{col_index}"] for col_index in range(IMAGE_SIZE)] for row_index in range(IMAGE_SIZE)]


def flatten(matrix):
    return [value for line in matrix for value in line]


def ascii_heatmap(image):
    shades = " .:-=+*#%@"
    maximum = max(flatten(image)) or 1.0
    lines = []
    for line in image:
        chars = []
        for value in line:
            index = int(round((len(shades) - 1) * value / maximum))
            chars.append(shades[index])
        lines.append("".join(chars))
    return "\n".join(lines)


print(f"Loaded {len(rows)} cutout samples from {DATA_PATH.name}")
print("class counts:", dict(Counter(row["morphology_label"] for row in rows)))
print("train/test sizes:", len(train_rows), len(test_rows))
print("test ids:", [row["sample_id"] for row in test_rows])
print("Example cutouts:")
for sample_id in ["C01", "E01", "D01"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    print(f"\n{sample_id} {row['morphology_label']}")
    print(ascii_heatmap(image_from_row(row)))


In [ ]:
import math


def centroid(vectors):
    return [sum(values) / len(values) for values in zip(*vectors)]


def euclidean_distance(left, right):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(left, right)))


def nearest_centroid(train_items, feature_fn, test_items):
    labels = sorted({item["morphology_label"] for item in train_items})
    centers = {}
    for label in labels:
        vectors = [feature_fn(item) for item in train_items if item["morphology_label"] == label]
        centers[label] = centroid(vectors)

    predictions = []
    for item in test_items:
        vector = feature_fn(item)
        ranked = sorted(
            ((label, euclidean_distance(vector, centers[label])) for label in labels),
            key=lambda pair: pair[1],
        )
        predictions.append({
            "sample_id": item["sample_id"],
            "true_label": item["morphology_label"],
            "predicted_label": ranked[0][0],
            "distances": {label: round(distance, 3) for label, distance in ranked},
        })
    return centers, predictions


def accuracy(predictions):
    return sum(item["true_label"] == item["predicted_label"] for item in predictions) / len(predictions)


def confusion(predictions):
    matrix = {}
    for item in predictions:
        truth = item["true_label"]
        predicted = item["predicted_label"]
        matrix.setdefault(truth, {})
        matrix[truth][predicted] = matrix[truth].get(predicted, 0) + 1
    return matrix


raw_centers, raw_predictions = nearest_centroid(
    train_rows,
    lambda item: flatten(image_from_row(item)),
    test_rows,
)

print("Raw-pixel nearest-centroid baseline:")
for item in raw_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("raw accuracy:", round(accuracy(raw_predictions), 3))
print("raw confusion matrix:", confusion(raw_predictions))


In [ ]:
kernels = {
    "core": [
        [-1.0, -1.0, -1.0],
        [-1.0, 6.0, -1.0],
        [-1.0, -1.0, -1.0],
    ],
    "horizontal": [
        [-1.0, -1.0, -1.0],
        [1.5, 1.5, 1.5],
        [-1.0, -1.0, -1.0],
    ],
    "double_lobe": [
        [1.0, -1.0, 1.0],
        [1.0, -3.0, 1.0],
        [1.0, -1.0, 1.0],
    ],
}


def conv2d_valid(image, kernel):
    output = []
    for row_index in range(len(image) - 2):
        row = []
        for col_index in range(len(image[0]) - 2):
            score = 0.0
            for kernel_row in range(3):
                for kernel_col in range(3):
                    score += image[row_index + kernel_row][col_index + kernel_col] * kernel[kernel_row][kernel_col]
            row.append(score)
        output.append(row)
    return output


def relu(matrix):
    return [[max(0.0, value) for value in line] for line in matrix]


def max_pool_2x2(feature_map):
    pooled = []
    for row_index in range(0, len(feature_map) - 1, 2):
        row = []
        for col_index in range(0, len(feature_map[0]) - 1, 2):
            row.append(max(
                feature_map[row_index][col_index],
                feature_map[row_index][col_index + 1],
                feature_map[row_index + 1][col_index],
                feature_map[row_index + 1][col_index + 1],
            ))
        pooled.append(row)
    return pooled


def summarize_kernel_response(image):
    summary = []
    detailed = {}
    for kernel_name, kernel in kernels.items():
        pooled = max_pool_2x2(relu(conv2d_valid(image, kernel)))
        pooled_values = flatten(pooled)
        summary.extend([max(pooled_values), sum(pooled_values) / len(pooled_values)])
        detailed[kernel_name] = pooled
    return summary, detailed


print("Pooled responses for representative training samples:")
for sample_id in ["C01", "E01", "D01"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    feature_summary, detailed = summarize_kernel_response(image_from_row(row))
    print(f"\n{sample_id} {row['morphology_label']}")
    print("feature summary:", [round(value, 2) for value in feature_summary])
    for kernel_name, pooled in detailed.items():
        print(kernel_name, [[round(value, 2) for value in line] for line in pooled])


In [ ]:
def cnn_features(item):
    summary, _ = summarize_kernel_response(image_from_row(item))
    return summary


conv_centers, conv_predictions = nearest_centroid(train_rows, cnn_features, test_rows)

print("Convolution-feature nearest-centroid model:")
for item in conv_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("conv accuracy:", round(accuracy(conv_predictions), 3))
print("conv confusion matrix:", confusion(conv_predictions))
print("accuracy improvement over raw baseline:", round(accuracy(conv_predictions) - accuracy(raw_predictions), 3))


In [ ]:
print("Translation check on the double-source class:")
for sample_id in ["D01", "D03", "D04"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    summary, detailed = summarize_kernel_response(image_from_row(row))
    print(sample_id, row["split"], [round(value, 2) for value in summary])
    print("double_lobe pooled =", [[round(value, 2) for value in line] for line in detailed["double_lobe"]])

print("Interpretation:")
print("  The raw-pixel baseline memorizes exact coordinates, so shifted double-source samples drift toward the edge-on centroid.")
print("  Convolution keeps looking for the same local motif at different positions, and max pooling keeps the strongest response.")
print("  That is the first practical reason CNN-style features outperform flattened pixels on spatial data.")


In [ ]:
TRANSFER_DATA_PATH = Path("../../data/small/cnn_transfer_learning_demo.csv").resolve()

with TRANSFER_DATA_PATH.open(newline="", encoding="utf-8") as handle:
    transfer_rows = []
    for raw in csv.DictReader(handle):
        parsed = {
            "sample_id": raw["sample_id"],
            "morphology_label": raw["morphology_label"],
            "domain_label": raw["domain_label"],
            "split_role": raw["split_role"],
        }
        for row_index in range(IMAGE_SIZE):
            for col_index in range(IMAGE_SIZE):
                parsed[f"p{row_index}{col_index}"] = float(raw[f"p{row_index}{col_index}"])
        transfer_rows.append(parsed)

source_train_rows = [row for row in transfer_rows if row["split_role"] == "source_train"]
target_adapt_rows = [row for row in transfer_rows if row["split_role"] == "target_adapt"]
target_test_rows = [row for row in transfer_rows if row["split_role"] == "target_test"]

print(f"Loaded {len(transfer_rows)} transfer-learning samples from {TRANSFER_DATA_PATH.name}")
print("split-role counts:", dict(Counter(row["split_role"] for row in transfer_rows)))
print("target-test ids:", [row["sample_id"] for row in target_test_rows])
print("Example source vs target cutouts:")
for sample_id in ["S_C1", "T_CA", "T_E1"]:
    row = next(item for item in transfer_rows if item["sample_id"] == sample_id)
    print(f"\n{sample_id} {row['domain_label']} {row['morphology_label']}")
    print(ascii_heatmap(image_from_row(row)))


In [ ]:
def transfer_report(name, train_items, feature_fn):
    _, predictions = nearest_centroid(train_items, feature_fn, target_test_rows)
    print(f"\n{name}")
    for item in predictions:
        print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
    print("accuracy:", round(accuracy(predictions), 3))
    print("confusion matrix:", confusion(predictions))
    return predictions


source_raw_transfer_predictions = transfer_report(
    "Source raw-pixel prototypes -> target test",
    source_train_rows,
    lambda item: flatten(image_from_row(item)),
)
source_conv_transfer_predictions = transfer_report(
    "Source frozen conv features -> target test",
    source_train_rows,
    cnn_features,
)
adapt_raw_transfer_predictions = transfer_report(
    "One-shot target raw head -> target test",
    target_adapt_rows,
    lambda item: flatten(image_from_row(item)),
)
adapt_conv_transfer_predictions = transfer_report(
    "Frozen conv features + one-shot target head -> target test",
    target_adapt_rows,
    cnn_features,
)

transfer_metrics = {
    "source raw-pixel prototypes": round(accuracy(source_raw_transfer_predictions), 3),
    "source frozen conv features": round(accuracy(source_conv_transfer_predictions), 3),
    "one-shot target raw head": round(accuracy(adapt_raw_transfer_predictions), 3),
    "frozen conv features + one-shot target head": round(accuracy(adapt_conv_transfer_predictions), 3),
}
print("\nTransfer-learning style summary:")
for label, metric in transfer_metrics.items():
    print(label, "->", metric)

print("Interpretation:")
print("  Raw pixels drift when the target survey changes background level and exact coordinates.")
print("  The frozen convolution-style features still capture compact cores, horizontal disks, and double sources.")
print("  That is the notebook's smallest transfer-learning lesson: keep the reusable feature extractor, then adapt only a lightweight head.")


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")
else:
    figure, axes = plt.subplots(2, 3, figsize=(8, 5))
    for axis, sample_id in zip(axes.flat, ["C01", "E01", "D01", "C03", "E03", "D03"]):
        row = next(item for item in rows if item["sample_id"] == sample_id)
        axis.imshow(image_from_row(row), cmap="magma", interpolation="nearest")
        axis.set_title(f"{sample_id}\n{row['morphology_label']}")
        axis.set_xticks([])
        axis.set_yticks([])
    figure.suptitle("Tiny galaxy-cutout teaching samples")
    figure.tight_layout()
    plt.show()


In [ ]:
try:
    import torch
except ModuleNotFoundError as exc:
    print(f"Optional PyTorch comparison skipped: {exc}")
else:
    torch.manual_seed(0)
    label_to_index = {label: index for index, label in enumerate(sorted({row["morphology_label"] for row in rows}))}
    x_train = torch.tensor([image_from_row(row) for row in train_rows], dtype=torch.float32).unsqueeze(1)
    y_train = torch.tensor([label_to_index[row["morphology_label"]] for row in train_rows], dtype=torch.long)
    x_test = torch.tensor([image_from_row(row) for row in test_rows], dtype=torch.float32).unsqueeze(1)
    y_test = torch.tensor([label_to_index[row["morphology_label"]] for row in test_rows], dtype=torch.long)

    model = torch.nn.Sequential(
        torch.nn.Conv2d(1, 4, kernel_size=3),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Flatten(),
        torch.nn.Linear(4, len(label_to_index)),
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
    loss_fn = torch.nn.CrossEntropyLoss()

    for _ in range(300):
        optimizer.zero_grad()
        logits = model(x_train)
        loss = loss_fn(logits, y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        test_logits = model(x_test)
        predicted = test_logits.argmax(dim=1)
        test_accuracy = (predicted == y_test).float().mean().item()
        reverse_labels = {index: label for label, index in label_to_index.items()}
        print("Optional PyTorch tiny-CNN accuracy:", round(test_accuracy, 3))
        for row, pred_index in zip(test_rows, predicted.tolist()):
            print("torch", row["sample_id"], row["morphology_label"], "->", reverse_labels[pred_index])
